In [1]:
using Bloqade
using MLDatasets: MNIST
using MultivariateStats: fit, PCA, transform
using LinearAlgebra
using Statistics
using Random
using LinearAlgebra
#------------------------
λ = 1  # 编码尺度           
d0 = 10     # 初始原子间距                           
const C = 862690.0 * 2π      
omega = 2π * 1.0        
delta_g = 2π * 1.0       
V0 = C / d0^6
#-------------------------
function prepare_all_data(; train_samples=500, test_samples=50, seed=42)
    Random.seed!(seed)
    println("---加载 MNIST 数据---")
    
    x_train, y_train = MNIST.traindata()
    x_test, y_test = MNIST.testdata()
    
    # 展平 (784, N)
    X = float.(reshape(x_train, 784, :))
    Y = y_train
    X_test = float.(reshape(x_test, 784, :))
    Y_test = y_test

    # 筛选目标类别 (3 和 8)
    println("---筛选目标类别 (3 和 8)---")
    train_mask = (Y .== 3) .| (Y .== 8)
    test_mask = (Y_test .== 3) .| (Y_test .== 8)
    
    X_target = X[:, train_mask]
    Y_target = Y[train_mask]
    X_test_target = X_test[:, test_mask]
    Y_test_target = Y_test[test_mask]

    # 对目标类别进行PCA拟合
    println("---PCA 拟合 (基于目标类别 3 和 8)---")
    M = fit(PCA, X_target; maxoutdim=8) 
    p_vars = principalvars(M) 
    total_variance = sum(p_vars) + tprincipalvar(M) 
    variance_ratio = p_vars ./ total_variance
    println("前 8 个主成分方差贡献率：", variance_ratio)
    
    # 转换数据
    X_tol = transform(M, X_target)
    test_data = transform(M, X_test_target)
    min_val = minimum(X_tol)
    max_val = maximum(X_tol)
    std_all = std(X_tol)
    println("X_tol 的最小值: ", min_val)
    println("X_tol 的最大值: ", max_val)
    println("X_tol 的整体标准差: ", std_all)
    test_min = minimum(test_data)
    test_max = maximum(test_data)
    println("test 的最小值: ", test_min)
    println("test 的最大值: ", test_max)

    
    # 标签映射为0和1
    Y_binary = [y == 8 ? 1 : 0 for y in Y_target]
    Y_test_binary = [y == 8 ? 1 : 0 for y in Y_test_target]
    
    # 选择特定数量的样本
    n_total = size(X_tol, 2)
    n_train = min(train_samples, n_total)
    sample_indices = randperm(n_total)[1:n_train]
    train = X_tol[:, sample_indices]
    Y_binary = Y_binary[sample_indices]
    # 测试集
    n_test_total = size(test_data, 2)
    n_test = min(test_samples, n_test_total)
    test_sample_indices = randperm(n_test_total)[1:n_test]
    test = test_data[:, test_sample_indices]
    Y_test_binary = Y_test_binary[test_sample_indices]

    #平移缩放
    train = (train .- min_val)/(max_val - min_val)
    test = (test .- min_val)/(max_val - min_val)
    
    # 返回筛选后的数据
    return train, Y_binary, test, Y_test_binary
end

Precompiling packages...
Info Given DiffEqBaseChainRulesCoreExt was explicitly requested, output will be shown live 
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
   2256.7 ms  ? DiffEqBase → DiffEqBaseChainRulesCoreExt
[ Info: Precompiling DiffEqBaseChainRulesCoreExt [b00db79b-61e3-50fb-b26f-2d35b2d9e4ed]
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
[ Info: Skipping precompilation since __precompile__(false). Importing DiffEqBaseChainRulesCoreExt [b00db79b-61e3-50fb-b26f-2d35b2d9e4ed].


prepare_all_data (generic function with 1 method)

In [2]:
using MultivariateStats
X, y, X_test, y_test = prepare_all_data(train_samples=50000,test_samples = 5000)

println("训练集矩阵形状: ", size(X)) 
println("训练集标签向量长度: ", length(y)) 
println("测试集矩阵形状: ", size(X_test)) 
println("测试集标签向量长度: ", length(y_test)) 
println("标签内容 (应全为0或1): ", unique(y)) 
println("前5个标签: ", y[1:5])
println("前5列数据") 
display(X[:, 1:5]) 

---加载 MNIST 数据---


┌ Warning: MNIST.traindata() is deprecated, use `MNIST(split=:train)[:]` instead.
└ @ MLDatasets C:\Users\26676\.julia\packages\MLDatasets\XUXth\src\datasets\vision\mnist.jl:187
┌ Warning: MNIST.testdata() is deprecated, use `MNIST(split=:test)[:]` instead.
└ @ MLDatasets C:\Users\26676\.julia\packages\MLDatasets\XUXth\src\datasets\vision\mnist.jl:195


---筛选目标类别 (3 和 8)---
---PCA 拟合 (基于目标类别 3 和 8)---
前 8 个主成分方差贡献率：Float32[0.13884974, 0.08714235, 0.07505022, 0.056729738, 0.04505979, 0.035908647, 0.032081716, 0.029177802]
X_tol 的最小值: -5.5883856
X_tol 的最大值: 6.838207
X_tol 的整体标准差: 1.6477185
test 的最小值: -5.3230653
test 的最大值: 6.85183
训练集矩阵形状: (8, 11982)
训练集标签向量长度: 11982
测试集矩阵形状: (8, 1984)
测试集标签向量长度: 1984
标签内容 (应全为0或1): [0, 1]
前5个标签: [0, 1, 1, 0, 1]
前5列数据


8×5 Matrix{Float32}:
 0.186207  0.322241  0.492379  0.369277  0.719916
 0.271299  0.509181  0.518481  0.262573  0.349697
 0.582225  0.364934  0.319353  0.236064  0.446499
 0.452054  0.605871  0.680291  0.441597  0.57884
 0.518461  0.633695  0.576409  0.526779  0.552453
 0.41226   0.469873  0.416293  0.396832  0.527304
 0.408288  0.432111  0.380955  0.362167  0.406155
 0.451099  0.364359  0.485935  0.329754  0.601555

In [3]:
train = X'
println("训练集形状: ", size(train))
test = X_test'
println("测试集形状: ", size(test))
println("y:",size(y))
println("y_test",size(y_test))

训练集形状: (11982, 8)
测试集形状: (1984, 8)
y:(11982,)
y_test(1984,)


In [4]:
using CSV, DataFrames
CSV.write("mnist_train.csv", DataFrame(train, :auto), header=false)
CSV.write("mnist_test.csv", DataFrame(test, :auto), header=false)
CSV.write("mnist_y_train.csv", DataFrame([y], :auto), header=false)
CSV.write("mnist_y_test.csv", DataFrame([y_test], :auto), header=false)

"mnist_y_test.csv"

In [7]:
function encode(features::Vector{Float64},runtime::AbstractFloat) 
    alpha = 1 #加权scaling
if length(features) != n_components
        error("输入特征长度必须等于 n_components ($n_components)") 
end
    positions_raw = zeros(Float64, n_atoms)
    positions_raw[1] = 0.0        
# 计算理论位置    
for i in 1:n_components
        xi = features[i]
        V_target = V0 * (1.0 + λ * xi)
        r = (C / V_target)^(1/6)
        positions_raw[i+1] = positions_raw[i] + r    
end
# 演化
atom_coordinates =
    [(positions_raw[i], 0.0) for i in 1:n_atoms]

reg = zero_state(n_atoms)

h = rydberg_h(
    atom_coordinates;
    Ω = omega,
    Δ = delta_g
)

prob = SchrodingerProblem(reg, runtime, h)
emulate!(prob)

# 单体观测
single = zeros(Float64, n_atoms)
z_expect = zeros(Float64, n_atoms)

for i in 1:n_atoms
    single[i] = rydberg_density(prob.reg, i)
    z_expect[i] = 2 * single[i] - 1
end


# 两体关联
pair = zeros(Float64, n_atoms - 1)
for i in 1:(n_atoms - 1)
    op =
        put(n_atoms, i => Op.n) * put(n_atoms, (i+1) => Op.n)
    pair[i] =
        real(expect(op, prob.reg))
end
for i in 1:(n_atoms - 1) # connected correlation function
    pair[i] = ( 4*pair[i] -2*single[i] - 2*single[i+1] + 1- z_expect[i]*z_expect[i+1] ) * alpha
end

output = vcat(z_expect, pair)

    return output
end

encode (generic function with 1 method)

In [8]:
## 量子编码（位置编码9维）
t = 0.5
using Printf

n_components = 8
n_atoms = n_components + 1
println("\n--- 物理参数 ---")
println("初始距离 d0: $d0 µm")
println("编码尺度 λ: $λ")

X0 = train'

num_tests=size(train,1)
results = zeros(Float64, 2*n_atoms-1, num_tests)
for col_idx in 1:num_tests
    features = Vector{Float64}(X0[:, col_idx])
    try
        pops = encode(features, t)
        results[:, col_idx] = pops
    catch e
        println("❌ 第 $col_idx 列模拟失败: ", e)
    end
end
println("量子编码矩阵形状: ", size(results))
println("前5列数据") 
display(results[:, 1:5]) 


--- 物理参数 ---
初始距离 d0: 10 µm
编码尺度 λ: 1
量子编码矩阵形状: (17, 11982)
前5列数据


17×5 Matrix{Float64}:
 0.361845  0.386608  0.401965  0.407791  0.389692
 0.496616  0.398446  0.35474   0.467186  0.327482
 0.3504    0.349492  0.355307  0.473312  0.370671
 0.296132  0.308133  0.289899  0.412704  0.298675
 0.316076  0.232342  0.227042  0.32146   0.263899
 0.330864  0.269307  0.306248  0.332278  0.281531
 0.367759  0.341136  0.374741  0.390428  0.32602
 0.398605  0.412814  0.396876  0.445315  0.352245
 0.408558  0.398242  0.411641  0.396413  0.40506
 0.293635  0.320824  0.372769  0.368223  0.481287
 0.236966  0.305131  0.267323  0.286983  0.148571
 0.257124  0.125397  0.100508  0.225049  0.180586
 0.126075  0.153418  0.186901  0.230653  0.158869
 0.204743  0.127932  0.120739  0.214025  0.123224
 0.180474  0.138534  0.171731  0.190707  0.173344
 0.240621  0.250401  0.227594  0.272401  0.162669
 0.37676   0.342675  0.393711  0.339478  0.427128

In [9]:
# 储存到ministtrain17.csv
using CSV, DataFrames
#col_names = ["Feature_$i" for i in 1:size(data_matrix, 2)]
# 转换为 DataFrame
data = Matrix(results')
df = DataFrame(data,:auto)
# 保存为 CSV
CSV.write("mnisttrain17.csv", df,header=false)
println("文件已保存")

文件已保存


In [10]:
## 量子编码（位置编码,测试集）
using Printf

t = 0.5
n_components = 8
n_atoms = n_components + 1
println("\n--- 物理参数 ---")
println("初始距离 d0: $d0 µm")
println("编码尺度 λ: $λ")

X0=test'

num_tests=size(test,1)
results = zeros(Float64, 2*n_atoms-1, num_tests)
for col_idx in 1:num_tests
    features = Vector{Float64}(X0[:, col_idx])
    try
        pops = encode(features, t)
        results[:, col_idx] = pops
    catch e
        println("❌ 第 $col_idx 列模拟失败: ", e)
    end
end
println("量子编码矩阵形状: ", size(results))
println("前5列数据") 
display(results[:, 1:5]) 


--- 物理参数 ---
初始距离 d0: 10 µm
编码尺度 λ: 1
量子编码矩阵形状: (17, 1984)
前5列数据


17×5 Matrix{Float64}:
 0.349808   0.380206   0.401745  0.41617   0.364383
 0.249315   0.319939   0.357208  0.442893  0.269707
 0.252504   0.264337   0.358104  0.490601  0.303266
 0.276397   0.28003    0.295057  0.454032  0.402896
 0.299847   0.307262   0.339425  0.337367  0.399205
 0.349941   0.393631   0.32777   0.334188  0.357006
 0.349939   0.336681   0.318617  0.32445   0.370487
 0.366506   0.353713   0.414713  0.372464  0.42793
 0.40307    0.402736   0.390765  0.385254  0.389875
 0.41665    0.310585   0.500322  0.400932  0.423961
 0.0923265  0.279834   0.120498  0.290192  0.134757
 0.121562   0.0616395  0.249257  0.21961   0.218992
 0.141674   0.249299   0.167965  0.246057  0.170274
 0.19189    0.138872   0.153305  0.243481  0.276828
 0.159191   0.207809   0.2083    0.125388  0.189751
 0.256707   0.235231   0.242186  0.302499  0.287137
 0.367714   0.374495   0.326565  0.318339  0.326564

In [11]:
# 储存到mnisttest17.csv
#col_names = ["Feature_$i" for i in 1:size(data_matrix, 2)]
using CSV, DataFrames
# 转换为 DataFrame
data = Matrix(results')
df = DataFrame(data,:auto)

# 保存为 CSV
CSV.write("mnisttest17.csv", df,header=false)

println("文件已保存")

文件已保存


In [12]:
## 量子编码（位置编码2，训练集）
using Printf

n_components = 8
println("\n--- 物理参数 ---")
println("初始距离 d0: $d0 µm")
println("编码尺度 λ: $λ")

X0 = train'

num_tests=size(train,1)
results = zeros(Float64, 51, num_tests)
for col_idx in 1:num_tests
    features = Vector{Float64}(X0[:, col_idx])
    try
        pops_05 = encode(features, 0.5)
        pops_10 = encode(features, 1.0)
        pops_15 = encode(features, 1.5)
        # 垂直拼接成一个长向量
        pops = vcat(pops_05, pops_10, pops_15)
        results[:, col_idx] = pops
    catch e
        println("❌ 第 $col_idx 列模拟失败: ", e)
    end
end
println("量子编码矩阵形状: ", size(results))
println("前5列数据")
display(results[:, 1:5])


--- 物理参数 ---
初始距离 d0: 10 µm
编码尺度 λ: 1
量子编码矩阵形状: (51, 11982)
前5列数据


51×5 Matrix{Float64}:
  0.361845    0.386608    0.401965    0.407791    0.389692
  0.496616    0.398446    0.35474     0.467186    0.327482
  0.3504      0.349492    0.355307    0.473312    0.370671
  0.296132    0.308133    0.289899    0.412704    0.298675
  0.316076    0.232342    0.227042    0.32146     0.263899
  0.330864    0.269307    0.306248    0.332278    0.281531
  0.367759    0.341136    0.374741    0.390428    0.32602
  0.398605    0.412814    0.396876    0.445315    0.352245
  0.408558    0.398242    0.411641    0.396413    0.40506
  0.293635    0.320824    0.372769    0.368223    0.481287
  0.236966    0.305131    0.267323    0.286983    0.148571
  0.257124    0.125397    0.100508    0.225049    0.180586
  0.126075    0.153418    0.186901    0.230653    0.158869
  ⋮                                              
  0.264975    0.274736    0.260224    0.234917    0.272866
 -0.0196978   0.100759    0.0829294   0.0185623   0.0408652
 -0.181922   -0.0844613  -0.148714   -0.1475

In [13]:
# 储存到mnisttrain51.csv
#col_names = ["Feature_$i" for i in 1:size(data_matrix, 2)]

# 转换为 DataFrame
data = Matrix(results')
df = DataFrame(data,:auto)

# 保存为 CSV
CSV.write("mnisttrain51.csv", df,header=false)

println("文件已保存")

文件已保存


In [14]:
## 量子编码（位置编码2，测试集）
using Printf

n_components = 8
println("\n--- 物理参数 ---")
println("初始距离 d0: $d0 µm")
println("编码尺度 λ: $λ")

X0 = test'

num_tests=size(test,1)
results = zeros(Float64, 51, num_tests)
for col_idx in 1:num_tests
    features = Vector{Float64}(X0[:, col_idx])
    try
        pops_05 = encode(features, 0.5)
        pops_10 = encode(features, 1.0)
        pops_15 = encode(features, 1.5)
        # 垂直拼接成一个长向量
        pops = vcat(pops_05, pops_10, pops_15)
        results[:, col_idx] = pops
    catch e
        println("❌ 第 $col_idx 列模拟失败: ", e)
    end
end
println("量子编码矩阵形状: ", size(results))
println("前5列数据")
display(results[:, 1:5])


--- 物理参数 ---
初始距离 d0: 10 µm
编码尺度 λ: 1
量子编码矩阵形状: (51, 1984)
前5列数据


51×5 Matrix{Float64}:
  0.349808    0.380206    0.401745    0.41617     0.364383
  0.249315    0.319939    0.357208    0.442893    0.269707
  0.252504    0.264337    0.358104    0.490601    0.303266
  0.276397    0.28003     0.295057    0.454032    0.402896
  0.299847    0.307262    0.339425    0.337367    0.399205
  0.349941    0.393631    0.32777     0.334188    0.357006
  0.349939    0.336681    0.318617    0.32445     0.370487
  0.366506    0.353713    0.414713    0.372464    0.42793
  0.40307     0.402736    0.390765    0.385254    0.389875
  0.41665     0.310585    0.500322    0.400932    0.423961
  0.0923265   0.279834    0.120498    0.290192    0.134757
  0.121562    0.0616395   0.249257    0.21961     0.218992
  0.141674    0.249299    0.167965    0.246057    0.170274
  ⋮                                              
  0.147236    0.170934    0.231409    0.203374    0.0815659
 -0.0210358  -0.0605811  -0.0220747  -0.0321792  -0.127888
 -0.205117   -0.253131   -0.139711   -0.188

In [15]:
# 储存到mnisttest51.csv
#col_names = ["Feature_$i" for i in 1:size(data_matrix, 2)]

# 转换为 DataFrame
data = Matrix(results')
df = DataFrame(data,:auto)

# 保存为 CSV
CSV.write("mnisttest51.csv", df,header=false)

println("文件已保存")

文件已保存
